# Advanced Meteorological Time Series Analysis for Rainfall Data

**Author**: PhD-level Meteorologist, Hydrologist, and Data Scientist

## Environment Verification
This notebook is designed to run in the `tensorflow` conda environment. It utilizes `pymannkendall` for trend analysis, `seaborn` for distribution plotting, and strictly adheres to meteorological aggregation rules (SUM).

## 1. Import Libraries & Setup Folder Structure

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pymannkendall as mk
from pathlib import Path
import warnings
import gc

warnings.filterwarnings('ignore')

output_dir = Path('output')
folders = [
    'figures/monthly', 'figures/annual', 'figures/climatology', 'figures/extremes', 'figures/distribution',
    'tables', 'reports'
]
for folder in folders:
    (output_dir / folder).mkdir(parents=True, exist_ok=True)

## Step 1: Data Loading & Inspection
Loading the CSV, parsing datetimes, and reporting temporal coverage including duplicates and missing intervals.

In [14]:
data_path = Path('../data/Rainfall_TimeSeries_Fixed_2005_2025.csv')
df = pd.read_csv(data_path)
dt_col = df.columns[0]
val_col = df.columns[1]

df[dt_col] = pd.to_datetime(df[dt_col])
df.set_index(dt_col, inplace=True)
df.sort_index(inplace=True)

# Missing & Duplicated
dup_count = df.index.duplicated().sum()
df = df[~df.index.duplicated(keep='first')]

expected_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='30min')
missing_timestamps = expected_idx.difference(df.index)

df = df.reindex(expected_idx)
df.rename(columns={val_col: 'rainfall'}, inplace=True)
missing_values_count = df['rainfall'].isna().sum()

coverage_report = pd.DataFrame({
    'Metric': ['Start Date', 'End Date', 'Total Records', 'Missing Timestamps', 'Missing Values', 'Duplicates Removed'],
    'Value': [str(df.index.min()), str(df.index.max()), len(df), len(missing_timestamps), missing_values_count, dup_count]
})
display(coverage_report)
coverage_report.to_csv(output_dir / 'reports' / 'temporal_coverage.csv', index=False)

,Metric,Value
0,Start Date,2005-01-01 00:00:00
1,End Date,2025-12-31 23:30:00
2,Total Records,368160
3,Missing Timestamps,216
4,Missing Values,24807
5,Duplicates Removed,0


## Step 2: Quality Control (QC)
Replacing unrealistic and negative values with `NaN`.

In [15]:
invalid_mask = (df['rainfall'] < 0) | np.isinf(df['rainfall'])
df.loc[invalid_mask, 'rainfall'] = np.nan

qc_summary = pd.DataFrame({
    'Metric': ['Negative Rainfall', 'Infinite Rainfall', 'Total Invalid/Missing'],
    'Count': [(df['rainfall'] < 0).sum(), np.isinf(df['rainfall']).sum(), df['rainfall'].isna().sum()]
})
display(qc_summary)
qc_summary.to_csv(output_dir / 'reports' / 'qc_summary.csv', index=False)

# Fill NaNs with 0 to safely sum() as non-observations represent dry periods in hydrology.
df_clean = df.copy()
df_clean['rainfall'] = df_clean['rainfall'].fillna(0.0)

,Metric,Count
0,Negative Rainfall,0
1,Infinite Rainfall,0
2,Total Invalid/Missing,24807


## Step 3: Time Aggregations
Strictly utilizing `.sum()` for volumetric mass conservation.

In [16]:
df_hourly = df_clean.resample('h').sum()
df_daily = df_clean.resample('D').sum()
df_monthly = df_clean.resample('MS').sum()
df_annual = df_clean.resample('YS').sum()

## Step 4: Entire Period Time Series

In [17]:
fig, ax = plt.subplots(figsize=(20, 6))
ax.plot(df_daily.index, df_daily['rainfall'], color='#3498db', linewidth=0.8)
ax.set_title('Rainfall Time Series (2005-2025)', fontsize=16)
ax.set_xlabel('Year')
ax.set_ylabel('Daily Rainfall Accumulation (mm)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'Rainfall_2005_2025.png', dpi=300)
fig.clf()
plt.close(fig)
gc.collect()

24928

## Step 5: Monthly Analysis
Generating an individual daily bar chart for every month.

In [ ]:
for year in df_monthly.index.year.unique():
    year_data = df_daily[df_daily.index.year == year]
    for month in year_data.index.month.unique():
        month_data = year_data[year_data.index.month == month]
        month_name = month_data.index[0].strftime('%B')
        
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(month_data.index, month_data['rainfall'], color='#2ecc71', width=0.8)
        ax.set_title(f'Rainfall Time Series\n{month_name} {year}', fontsize=14)
        ax.set_xlabel('Date')
        ax.set_ylabel('Daily Rainfall (mm)')
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d'))
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'figures' / 'monthly' / f'{month_name}_{year}.png', dpi=300)
        fig.clf()
        plt.close(fig)
    gc.collect()
print("Monthly Analysis charts automatically exported to output/figures/monthly/")

## Step 6: Annual Analysis
Generating an individual chart per year mapping daily accumulation.

In [ ]:
for year in df_annual.index.year.unique():
    year_data = df_daily[df_daily.index.year == year]
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.bar(year_data.index, year_data['rainfall'], color='#9b59b6', width=1.0)
    ax.set_title(f'Annual Daily Rainfall Accumulation - {year}', fontsize=14)
    ax.set_xlabel('Date')
    ax.set_ylabel('Daily Rainfall (mm)')
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'figures' / 'annual' / f'Rainfall_Annual_{year}.png', dpi=300)
    fig.clf()
    plt.close(fig)
    gc.collect()
print("Annual Analysis charts exported to output/figures/annual/")

Annual Analysis charts exported to output/figures/annual/


## Step 7: Climatology

In [ ]:
monthly_climatology = df_monthly.groupby(df_monthly.index.month).mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_climatology.index, monthly_climatology['rainfall'], color='#e67e22')
ax.set_title('Mean Monthly Rainfall Climatology', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Mean Monthly Rainfall (mm)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'climatology' / 'Monthly_Climatology.png', dpi=300)
fig.clf()
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_annual.index.year, df_annual['rainfall'], color='#34495e')
ax.set_title('Annual Rainfall Totals', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Total Rainfall (mm)')
ax.set_xticks(df_annual.index.year)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'climatology' / 'Annual_Rainfall_Totals.png', dpi=300)
fig.clf()
plt.close(fig)

## Step 8: Extreme Rainfall Analysis
Extracting exact max magnitudes.

In [ ]:
extremes = {
    'Resolution': ['30-minute', 'Hourly', 'Daily', 'Monthly', 'Annual'],
    'Maximum Magnitude (mm)': [
        df_clean['rainfall'].max(),
        df_hourly['rainfall'].max(),
        df_daily['rainfall'].max(),
        df_monthly['rainfall'].max(),
        df_annual['rainfall'].max()
    ],
    'Timestamp': [
        str(df_clean['rainfall'].idxmax()),
        str(df_hourly['rainfall'].idxmax()),
        str(df_daily['rainfall'].idxmax()),
        str(df_monthly['rainfall'].idxmax()),
        str(df_annual['rainfall'].idxmax())
    ]
}
extremes_df = pd.DataFrame(extremes)
extremes_df.to_csv(output_dir / 'tables' / 'extreme_rainfall_summary.csv', index=False)
display(extremes_df)

,Resolution,Maximum Magnitude (mm),Timestamp
0,30-minute,59.852886,2014-04-12 11:30:00
1,Hourly,59.852886,2014-04-12 11:00:00
2,Daily,234.655819,2020-03-04 00:00:00
3,Monthly,1332.536722,2024-12-01 00:00:00
4,Annual,8314.022520,2016-01-01 00:00:00


## Step 9: Trend Analysis (Mann-Kendall & Sen's Slope)

In [ ]:
monthly_trend = mk.original_test(df_monthly['rainfall'])
annual_trend = mk.original_test(df_annual['rainfall'])

trend_report = pd.DataFrame({
    'Aggregation': ['Monthly', 'Annual'],
    'Trend Direction': [monthly_trend.trend, annual_trend.trend],
    'P-Value': [monthly_trend.p, annual_trend.p],
    'Tau': [monthly_trend.Tau, annual_trend.Tau],
    'Sen Slope': [monthly_trend.slope, annual_trend.slope]
})
trend_report.to_csv(output_dir / 'reports' / 'mann_kendall_trend.csv', index=False)
display(trend_report)

,Aggregation,Trend Direction,P-Value,Tau,Sen Slope
0,Monthly,increasing,3.162593e-09,0.250300,1.163310
1,Annual,increasing,4.570206e-05,0.647619,222.833227


## Step 10: Distribution Analysis
Building advanced statistical representations using seaborn.

In [ ]:
sns.set_theme(style="whitegrid")

for name, data in zip(['30-minute', 'Daily', 'Monthly'], [df_clean['rainfall'], df_daily['rainfall'], df_monthly['rainfall']]):
    if name != 'Monthly':
        data = data[data > 0] # Filter dry periods for meaningful visualization in high resolution metrics
        
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{name} Rainfall Distribution', fontsize=16)
    
    sns.histplot(data, bins=50, kde=False, ax=axes[0, 0], color='blue')
    axes[0, 0].set_title('Histogram')
    
    sns.kdeplot(data, ax=axes[0, 1], fill=True, color='red')
    axes[0, 1].set_title('KDE Plot')
    
    sns.boxplot(x=data, ax=axes[1, 0], color='green')
    axes[1, 0].set_title('Boxplot')
    
    sns.violinplot(x=data, ax=axes[1, 1], color='purple')
    axes[1, 1].set_title('Violin Plot')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'figures' / 'distribution' / f'{name}_Distribution.png', dpi=300)
    fig.clf()
    plt.close(fig)

print("Distribution analysis charts generated successfully.")

Distribution analysis charts generated successfully.


## Step 11: Wet and Dry Statistics & Extreme Event Detection

In [ ]:
wet_dry = []
heavy_events = []

for year in df_annual.index.year:
    year_data = df_daily[df_daily.index.year == year]
    
    rain_days = (year_data['rainfall'] > 0).sum()
    dry_days = (year_data['rainfall'] == 0).sum()
    
    heavy = (year_data['rainfall'] >= 50).sum()
    v_heavy = (year_data['rainfall'] >= 100).sum()
    extreme = (year_data['rainfall'] >= 150).sum()
    
    wet_dry.append({'Year': year, 'Rain Days': rain_days, 'Dry Days': dry_days})
    heavy_events.append({'Year': year, 'Heavy (>=50)': heavy, 'Very Heavy (>=100)': v_heavy, 'Extreme (>=150)': extreme})

pd.DataFrame(wet_dry).to_csv(output_dir / 'tables' / 'wet_dry_statistics.csv', index=False)
pd.DataFrame(heavy_events).to_csv(output_dir / 'tables' / 'extreme_events_frequency.csv', index=False)

print("All advanced statistical metrics have been processed and automatically saved to output/tables/.")

All advanced statistical metrics have been processed and automatically saved to output/tables/.
